In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 6 Lab — Model Tuning and Validation
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Measuring model stability with k-fold cross-validation
- Fitting a `RandomForestClassifier` and comparing it to a single decision tree
- Tuning hyperparameters systematically with `GridSearchCV`
- Following the correct final-model evaluation protocol

**Estimated time:** 80–95 minutes

---

## The Dataset

This lab uses the same customer churn dataset from Week 5. Rebuilding it here lets you compare Week 6 results directly to Week 5 — you will see whether cross-validation, random forests, and tuning improve on what you built last week.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import classification_report, f1_score, recall_score

# ── Rebuild churn dataset (same seed as Week 5) ──────────────────────────────
np.random.seed(13)
n = 400

tenure       = np.random.randint(1, 73, n)
monthly_chg  = np.round(np.random.uniform(20, 120, n), 2)
num_products = np.random.randint(1, 6, n)
tech_support = np.random.choice([0, 1], n, p=[0.45, 0.55])
contract     = np.random.choice(
    ['Month-to-Month', 'One Year', 'Two Year'], n, p=[0.50, 0.30, 0.20]
)
segment      = np.random.choice(
    ['Residential', 'SMB', 'Enterprise'], n, p=[0.55, 0.30, 0.15]
)
contract_risk = np.where(contract == 'Month-to-Month', 0.35,
               np.where(contract == 'One Year', 0.10, 0.03))
segment_risk  = np.where(segment == 'Residential', 0.08,
               np.where(segment == 'SMB', 0.03, 0.00))
churn_prob = np.clip(
    0.45 - (tenure / 72) * 0.35
    + (monthly_chg - 20) / 100 * 0.20
    - tech_support * 0.10
    - (num_products - 1) / 4 * 0.08
    + contract_risk + segment_risk,
    0.02, 0.95
)
churned = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

df = pd.DataFrame({
    'tenure_months':    tenure,
    'monthly_charges':  monthly_chg,
    'num_products':     num_products,
    'tech_support':     tech_support,
    'contract_type':    contract,
    'customer_segment': segment,
    'churned':          churned,
})

# ── Preprocess (same as Week 5) ──────────────────────────────────────────────
X = df.drop(columns=['churned'])
y = df['churned']

ordinal_cols     = ['contract_type']
categorical_cols = ['customer_segment']
numerical_cols   = ['tenure_months', 'monthly_charges', 'num_products']
passthrough_cols = ['tech_support']

preprocessor = ColumnTransformer(transformers=[
    ('ord',  OrdinalEncoder(categories=[['Month-to-Month', 'One Year', 'Two Year']]),
             ordinal_cols),
    ('cat',  OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
             categorical_cols),
    ('num',  StandardScaler(), numerical_cols),
    ('pass', 'passthrough', passthrough_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'Dataset ready: {df.shape[0]} rows  |  Churn rate: {y.mean():.1%}')
print(f'Train: {X_train_processed.shape}  |  Test: {X_test_processed.shape}')

---
## Part 1 — Cross-Validation: Measuring Stability

A single train/test split gives one F1 score. Cross-validation gives `k` scores — one per fold — so you can see how much performance varies depending on which rows the model trains on. Low variance across folds means the model is stable. High variance is a warning sign.

In [ ]:
# 5-fold cross-validation on logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)

lr_scores = cross_val_score(
    lr, X_train_processed, y_train,
    cv=5,
    scoring='f1'
)

print('Logistic Regression — 5-fold CV F1:')
print(f'  Per fold:  {lr_scores.round(3)}')
print(f'  Mean F1:   {lr_scores.mean():.3f}')
print(f'  Std F1:    {lr_scores.std():.3f}')

---
### 🤖 ML Connection — What the Standard Deviation Tells You

The standard deviation of CV scores measures how sensitive the model is to the specific training data in each fold. A std of 0.02 means performance is consistent — the model generalizes similarly regardless of which subset it trains on. A std of 0.12 means the model is unstable — some folds produce a strong model, others produce a weak one. High std is often caused by a small dataset, class imbalance in some folds, or a model that is too complex. Always report both mean and std when presenting CV results.

---

### ✏️ Exercise 1.1 — Compare CV Stability Across Models

1. Run 5-fold CV with F1 scoring on the `DecisionTreeClassifier(max_depth=4)` from Week 5.
2. Print per-fold scores, mean, and std for both logistic regression and the decision tree.
3. Which model has lower variance (lower std)? Write a comment explaining what that means in practice.
4. Plot both sets of fold scores as a grouped bar chart — one group per fold, two bars per group.

In [ ]:
# Exercise 1.1

# 1. CV on decision tree
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_scores = cross_val_score(
    # your code here
)

# 2. Print comparison
print('Decision Tree — 5-fold CV F1:')
print(f'  Per fold: {tree_scores.round(3)}')
print(f'  Mean F1:  {tree_scores.mean():.3f}')
print(f'  Std F1:   {tree_scores.std():.3f}')

# 3. Which has lower variance?
# 

# 4. Grouped bar chart
x     = np.arange(5)
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, lr_scores,   width, label='Logistic Regression', color='#4575b4')
ax.bar(x + width/2, tree_scores, width, label='Decision Tree',        color='#d73027')
ax.axhline(lr_scores.mean(),   color='#4575b4', linewidth=1, linestyle='--')
ax.axhline(tree_scores.mean(), color='#d73027', linewidth=1, linestyle='--')
ax.set_xlabel('Fold')
ax.set_ylabel('F1 Score')
ax.set_title('5-Fold CV F1 — Logistic Regression vs. Decision Tree')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 2 — Random Forest

A random forest builds many decision trees on bootstrap samples of the training data and combines their predictions. Each tree is different because it trains on a different sample and considers a random subset of features at each split. The ensemble is more stable and typically more accurate than any single tree.

In [ ]:
# Fit random forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,     # trees grow fully — bagging handles overfitting
    random_state=42
)
rf.fit(X_train_processed, y_train)

y_pred_rf = rf.predict(X_test_processed)

print('Random Forest — Test Set:')
print(classification_report(y_test, y_pred_rf, target_names=['Retained', 'Churned']))

# Feature importances
importances = pd.DataFrame({
    'feature':    preprocessor.get_feature_names_out(),
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature importances:')
print(importances.to_string(index=False))

---
### 🤖 ML Connection — Why Forests Outperform Single Trees

A single decision tree is a high-variance model: change a few training rows and the tree structure can change dramatically. A random forest reduces this variance by averaging across 100 (or more) trees trained on different bootstrap samples. Each individual tree may overfit its sample — but the errors are uncorrelated across trees, so they largely cancel out. This is why you can use `max_depth=None` in a forest without the catastrophic overfitting you would see in a single unlimited tree.

---

### ✏️ Exercise 2.1 — CV on Random Forest and Three-Way Comparison

1. Run 5-fold CV on the `RandomForestClassifier` using `X_train_processed` and `y_train`. Print mean and std F1.
2. Build a summary table comparing all three models: Logistic Regression, Decision Tree, Random Forest. Include CV mean F1, CV std F1, and test set F1 for each.
3. Write a comment: based on all three metrics, which model would you deploy and why?

In [ ]:
# Exercise 2.1

# 1. CV on random forest
rf_scores = cross_val_score(
    # your code here
)
print('Random Forest — 5-fold CV F1:')
print(f'  Per fold: {rf_scores.round(3)}')
print(f'  Mean F1:  {rf_scores.mean():.3f}')
print(f'  Std F1:   {rf_scores.std():.3f}')

# 2. Summary table
lr_test_f1   = f1_score(y_test, lr.predict(X_test_processed))    # provided — do not change
tree_test_f1 = f1_score(y_test, tree.predict(X_test_processed))  # provided — do not change
rf_test_f1   = f1_score(y_test, y_pred_rf)                       # provided — do not change

summary = pd.DataFrame({
    'Model':       ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'CV Mean F1':  [lr_scores.mean(), tree_scores.mean(), rf_scores.mean()],
    'CV Std F1':   [lr_scores.std(),  tree_scores.std(),  rf_scores.std()],
    'Test F1':     [lr_test_f1, tree_test_f1, rf_test_f1],
}).round(3)

print('\nModel comparison:')
print(summary.to_string(index=False))

# 3. Which would you deploy?
# 

---
## Part 3 — Hyperparameter Tuning with GridSearchCV

`GridSearchCV` tests every combination of hyperparameter values you specify, using cross-validation to evaluate each combination. It returns the best combination and can refit the model on the full training set automatically.

In [ ]:
# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [5, 10, 20, None],
}

# Run grid search — 3 × 4 = 12 combinations × 5 folds = 60 fits
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    refit=True    # refit best model on full training set (default)
)
grid_search.fit(X_train_processed, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1:      {grid_search.best_score_:.3f}')

---
### 💼 Business Context — Tuning Cost vs. Gain

Grid search with 12 combinations takes seconds on this dataset. On a dataset with 1 million rows and a grid of 50 combinations, it could take hours. `RandomizedSearchCV` addresses this by sampling a fixed number of random combinations from the grid — typically finding 90–95% of the optimal performance in 10–20% of the time. For production models, `RandomizedSearchCV` with `n_iter=20–50` is usually the right tool; `GridSearchCV` is reserved for small grids where exhaustive search is fast.

---

### ✏️ Exercise 3.1 — Inspect and Extend the Grid Search

1. Print the full CV results as a DataFrame: `pd.DataFrame(grid_search.cv_results_)`. Show only the columns `params`, `mean_test_score`, and `std_test_score`, sorted by `mean_test_score` descending.
2. How many total model fits did the grid search perform? Show the calculation.
3. Add `min_samples_leaf` with values `[1, 3, 5]` to `param_grid` and re-run the search. Print the new best parameters and best CV F1. Did it improve?

In [ ]:
# Exercise 3.1

# 1. Full CV results
results_df = pd.DataFrame(grid_search.cv_results_)
display_cols = ['params', 'mean_test_score', 'std_test_score']
print(results_df[display_cols]
      .sort_values('mean_test_score', ascending=False)
      .round(4)
      .to_string(index=False))

# 2. Total fits
n_combinations = 
n_folds        = 5
total_fits     = 
print(f'\nGrid: {n_combinations} combinations × {n_folds} folds = {total_fits} total fits')

# 3. Extended grid with min_samples_leaf
param_grid_ext = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [5, 10, 20, None],
    'min_samples_leaf': # your values here
}

grid_search_ext = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_ext,
    cv=5, scoring='f1', n_jobs=-1
)
grid_search_ext.fit(X_train_processed, y_train)

print(f'\nExtended grid — best parameters: {grid_search_ext.best_params_}')
print(f'Extended grid — best CV F1:      {grid_search_ext.best_score_:.3f}')
print(f'Original grid — best CV F1:      {grid_search.best_score_:.3f}')
# Did it improve?
# 

---
## Part 4 — The Final Model Protocol

After tuning, follow this exact sequence:

1. Retrieve `best_estimator_` — already refit on the full training set by `GridSearchCV`.
2. Predict on the test set — this is the first and only time the test set is used for evaluation.
3. Report final metrics.

The test score is the honest estimate of real-world performance. If it is substantially lower than the best CV score, the tuning process overfit to the training data.

In [ ]:
# Retrieve the best model (already refit on full training set)
best_model = grid_search.best_estimator_

# Final evaluation on the test set — once only
y_pred_final = best_model.predict(X_test_processed)

print('Final model — Test Set Evaluation:')
print(classification_report(y_test, y_pred_final, target_names=['Retained', 'Churned']))

print(f'Best CV F1 (from grid search): {grid_search.best_score_:.3f}')
print(f'Final test F1:                 {f1_score(y_test, y_pred_final):.3f}')
print(f'Gap (CV - Test):               {grid_search.best_score_ - f1_score(y_test, y_pred_final):.3f}')

---
### 🤖 ML Connection — Interpreting the CV-to-Test Gap

A small gap (< 0.05) between best CV F1 and test F1 is normal — the CV estimate is slightly optimistic because it averages over folds that share training data. A large gap (> 0.10) suggests the grid search overfit: with enough hyperparameter combinations, you will eventually find a combination that happens to do well on the specific folds in your training data but does not generalize. The fix is to use a coarser grid, more CV folds, or a larger dataset.

---

### ✏️ Exercise 4.1 — End-to-End Summary

1. Use `grid_search.best_estimator_` to access feature importances and print the top 5 features.
2. Compare the final tuned model's test F1 to the untuned random forest from Part 2. Did tuning improve performance?
3. Compare the tuned random forest's recall for the Churned class to the logistic regression recall from Part 1. Which would you recommend for a retention campaign focused on catching as many churners as possible?
4. Write a 4–5 sentence executive summary of the modelling process and recommendation, as if presenting to a non-technical manager.

In [ ]:
# Exercise 4.1

# 1. Top 5 feature importances from the best model
best_importances = pd.DataFrame({
    'feature':    preprocessor.get_feature_names_out(),
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 5 features (tuned model):')
print(best_importances.head(5).to_string(index=False))

# 2. Tuned vs. untuned RF
print(f'\nUntuned RF test F1: {rf_test_f1:.3f}')
print(f'Tuned RF test F1:   {f1_score(y_test, y_pred_final):.3f}')
# Comment:
# 

# 3. Recall comparison
tuned_recall = recall_score(y_test, y_pred_final)
lr_recall    = recall_score(y_test, lr.predict(X_test_processed))
print(f'\nTuned RF recall:             {tuned_recall:.3f}')
print(f'Logistic Regression recall:  {lr_recall:.3f}')
# Recommendation:
# 

# 4. Executive summary
# 
# 
# 
# 
# 

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

### RandomizedSearchCV with a Broader Parameter Space

`GridSearchCV` on a large parameter space is slow. `RandomizedSearchCV` samples a fixed number of random combinations, finding near-optimal settings much faster.

1. Define a parameter distribution using `scipy.stats.randint` for `n_estimators` (50 to 500) and `min_samples_leaf` (1 to 15), and a list for `max_depth` ([3, 5, 10, 20, None]).
2. Run `RandomizedSearchCV` with `n_iter=20`, `cv=5`, F1 scoring.
3. Print the best parameters and best CV F1. Compare to the `GridSearchCV` result.
4. **Reflection:** `RandomizedSearchCV` evaluated 20 combinations; `GridSearchCV` evaluated 12. How is it possible that random search finds better parameters with more combinations in a larger space? What are the trade-offs?

In [ ]:
# Challenge — RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators':     randint(50, 500),
    'max_depth':        [3, 5, 10, 20, None],
    'min_samples_leaf': # your code here
}

rand_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=20,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
# your code here — fit, print best params and best score

# Comparison to GridSearchCV:
# 

# Reflection:
# 

---
## Week 6 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| Single split variance | One test score is one data point — cross-validation gives a stable average |
| `cross_val_score` | Runs k-fold CV in one call — use on training data only |
| CV std | Low std = consistent model; high std = unstable, sensitive to data |
| `RandomForestClassifier` | Ensemble of trees on bootstrap samples — lower variance than single tree |
| `n_estimators` | More trees = more stable; returns diminish above ~200 |
| `max_depth=None` in forests | Safe — bagging handles overfitting; set it to tune precision/recall |
| Feature importances | Averaged across all trees — more stable than single-tree importances |
| `GridSearchCV` | Exhaustive — every combination × k folds; use for small grids |
| `RandomizedSearchCV` | Samples n_iter combinations — use for large grids |
| `best_estimator_` | Already refit on full training set — use this for final predictions |
| CV-to-test gap | Small (< 0.05) is normal; large (> 0.10) means tuning overfit |
| Final protocol | Tune on train → `best_estimator_` → predict test → report once |

---
**Next week:** End-to-end pipelines with `sklearn.Pipeline` and communicating results with Plotly Express. You will wrap everything from Weeks 3–6 — preprocessing, model, tuning — into a single object that applies every step in the correct order automatically.